In [ ]:
import matplotlib; matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# --- SVM One-Shot Revision ---
data = load_breast_cancer()
X, y = data.data, data.target
sc = StandardScaler()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)
for kernel in ['linear', 'rbf']:
    m = SVC(kernel=kernel, random_state=42)
    m.fit(X_train, y_train)
    print(f'{kernel}: {accuracy_score(y_test, m.predict(X_test)):.4f}')

# VIVA NOTES:
# 1. Kernel trick maps data to high-dim space without explicit computation (RBF, poly, etc.)
# 2. C = regularization - high C means low bias, high variance (fits training more)
# 3. gamma (RBF) = inverse radius - high gamma means tight boundary (overfitting)
# 4. Always scale features before SVM (sensitive to feature magnitude)
# 5. SVMs maximize margin between classes using support vectors only
# 6. Multiclass SVM uses OvO (one-vs-one) by default in sklearn


### Q7
Consider Social_Network_Ads.csv dataset - (UserID, Gender, Age, EstimatedSalary, Purchased). Use Age and EstimatedSalary as input features and Purchased as target feature. Split test data set 30% of complete dataset. Build two models of support vector classifier in python using sklearn library, one for linear and another for RBF kernel with C and gamma parameters set. Predict test labels and print test accuracy.

In [ ]:
import matplotlib; matplotlib.use('Agg')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


In [ ]:
# Load dataset - use Age and EstimatedSalary as features, Purchased as target
df = pd.read_csv('datasets/Social_Network_Ads.csv')
print(df.head())
print(f'Shape: {df.shape}')
X = df[['Age', 'EstimatedSalary']].values
y = df['Purchased'].values


In [ ]:
# 70/30 split, then scale features (SVM is sensitive to scale)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')


In [ ]:
# Linear SVC
model_lin = SVC(kernel='linear', C=1, random_state=42)
model_lin.fit(X_train, y_train)
y_pred_lin = model_lin.predict(X_test)
print(f'Linear SVC Accuracy: {accuracy_score(y_test, y_pred_lin):.4f}')

# RBF SVC
model_rbf = SVC(kernel='rbf', C=1, gamma=0.1, random_state=42)
model_rbf.fit(X_train, y_train)
y_pred_rbf = model_rbf.predict(X_test)
print(f'RBF SVC Accuracy: {accuracy_score(y_test, y_pred_rbf):.4f}')


In [ ]:
# Decision boundary plot for both kernels
def plot_boundary(ax, model, X_train, y_train, title):
    h = 0.02
    x_min, x_max = X_train[:, 0].min() - 1, X_train[:, 0].max() + 1
    y_min, y_max = X_train[:, 1].min() - 1, X_train[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.4, cmap='coolwarm')
    ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap='coolwarm', edgecolors='k', s=30)
    ax.set_title(title)
    ax.set_xlabel('Age (scaled)')
    ax.set_ylabel('EstimatedSalary (scaled)')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_boundary(axes[0], model_lin, X_train, y_train, 'SVC - Linear Kernel')
plot_boundary(axes[1], model_rbf, X_train, y_train, 'SVC - RBF Kernel (C=1, gamma=0.1)')
plt.tight_layout()
plt.show()


### Q8
Develop a Support Vector Classifier to predict whether a tumor is malignant or benign based on 30 features computed from a digitized image of a fine needle aspirate (FNA) of a breast mass. The dataset contains 569 samples with binary labels indicating tumor type (breast_cancer dataset of sklearn). Load the dataset from sklearn. Split the dataset into training and testing sets. Train the SVC model using the training data. Visualize the results.

In [ ]:
import matplotlib; matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.decomposition import PCA


In [ ]:
# Load breast cancer dataset (569 samples, 30 features, binary: malignant/benign)
data = load_breast_cancer()
X, y = data.data, data.target
print(f'Shape: {X.shape}, Classes: {data.target_names}')


In [ ]:
# 80/20 split, scale features before SVM
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

model = SVC(kernel='rbf', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(classification_report(y_test, y_pred, target_names=data.target_names))


In [ ]:
# (a) Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - SVC RBF (Breast Cancer)')
plt.tight_layout()
plt.show()


In [ ]:
# (b) 2D PCA projection - predicted vs true labels
pca = PCA(n_components=2, random_state=42)
X_test_2d = pca.fit_transform(X_test)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, labels, title in zip(axes, [y_test, y_pred], ['True Labels', 'Predicted Labels']):
    sc_plot = ax.scatter(X_test_2d[:, 0], X_test_2d[:, 1], c=labels,
                         cmap='coolwarm', edgecolors='k', s=50, alpha=0.8)
    ax.set_title(f'PCA Projection - {title}')
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(sc_plot, ax=ax, ticks=[0, 1])
plt.suptitle('SVC RBF - Breast Cancer (0=malignant, 1=benign)')
plt.tight_layout()
plt.show()


### Q9
Implement an SVC model to classify iris flowers into three species (Setosa, Versicolor, and Virginica) based on their sepal and petal dimensions. The dataset contains 150 samples with four features: sepal length, sepal width, petal length, and petal width. Load the dataset from sklearn. Split the dataset into training and testing sets. Train the SVC model using the training data. Visualize the results.

In [ ]:
import matplotlib; matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix


In [ ]:
# Load Iris dataset (150 samples, 4 features, 3 classes)
data = load_iris()
X, y = data.data, data.target
print(f'Shape: {X.shape}, Classes: {data.target_names}')


In [ ]:
# 80/20 split, scale features
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

model = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')


In [ ]:
# (a) Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=data.target_names, yticklabels=data.target_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - SVC RBF (Iris)')
plt.tight_layout()
plt.show()


In [ ]:
# (b) Decision boundary using first 2 features (sepal length, sepal width) - unscaled range
X2 = data.data[:, :2]  # sepal length and sepal width only
y2 = data.target

X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)
sc2 = StandardScaler()
X2_train_s = sc2.fit_transform(X2_train)

m2 = SVC(kernel='rbf', C=1, gamma='scale', random_state=42)
m2.fit(X2_train_s, y2_train)

# Meshgrid over unscaled iris sepal range
x_min, x_max = X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5
y_min, y_max = X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))
grid_scaled = sc2.transform(np.c_[xx.ravel(), yy.ravel()])
Z = m2.predict(grid_scaled).reshape(xx.shape)

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, Z, alpha=0.4, cmap='RdYlGn')
scatter = plt.scatter(X2[:, 0], X2[:, 1], c=y2, cmap='RdYlGn', edgecolors='k', s=40)
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Sepal Width (cm)')
plt.title('SVC Decision Boundary - Iris (Sepal Features)')
plt.colorbar(scatter, ticks=[0, 1, 2])
plt.tight_layout()
plt.show()
